In [ ]:
# 1. Login (Do this once)
from huggingface_hub import login
from huggingface_hub import create_repo, upload_folder
login("#hf_secret")

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
OUTPUT_DIR_25 = "/kaggle/working/wanda/Qwen3-4B-Instruct-2507-SGPT-25"
REPO_ID_25 = f"EdgeCompress01/Qwen3-4B-Instruct-2507-SGPT-25"

OUTPUT_DIR_50 = "/kaggle/working/wanda/Qwen3-4B-Instruct-2507-SGPT-50"
REPO_ID_50 = f"EdgeCompress01/Qwen3-4B-Instruct-2507-SGPT-50"

OUTPUT_DIR_70 = "/kaggle/working/wanda/Qwen3-4B-Instruct-2507-SGPT-70"
REPO_ID_70 = f"EdgeCompress01/Qwen3-4B-Instruct-2507-SGPT-70"
SEED = 42

# --- Configuration ---
NSAMPLES = 64  # Calibration samples
SEQ_LEN = 2048

# QWEN3

In [ ]:
!pip install -U "transformers>=5.1.0" "accelerate>=1.1.0" "datasets>=3.0.0"
!git clone https://github.com/locuslab/wanda.git
%cd wanda

In [ ]:
import os

file_path = '/kaggle/working/wanda/lib/data.py'

with open(file_path, 'r') as f:
    content = f.read()

# Remove the legacy config name 'allenai--c4'
fixed_content = content.replace("'allenai/c4', 'allenai--c4'", "'allenai/c4'")

with open(file_path, 'w') as f:
    f.write(fixed_content)

print("Fixed lib/data.py successfully!")

In [ ]:
# # Install/Update the required quantization libraries
# !pip install -U bitsandbytes accelerate

## Change the Main file to have the wanda only

In [ ]:
%%writefile main.py
import argparse
import os 
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import prune_sparsegpt, check_sparsity

def get_llm(model_id, cache_dir):
    # Standard float16, but load to CPU first to save GPU memory
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16,
        cache_dir=cache_dir, 
        low_cpu_mem_usage=True, 
        device_map="cpu" 
    )
    model.seqlen = 2048 
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, required=True)
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--nsamples', type=int, default=128)
    parser.add_argument('--sparsity_ratio', type=float, default=0.25)
    parser.add_argument("--cache_dir", default="llm_weights", type=str )
    parser.add_argument('--save_model', type=str, default=None)
    
    args = parser.parse_args()
    np.random.seed(args.seed)
    torch.random.manual_seed(args.seed)

    model = get_llm(args.model, args.cache_dir)
    tokenizer = AutoTokenizer.from_pretrained(args.model, use_fast=False)
    device = torch.device("cuda:0")
    
    print("Pruning starts (SparseGPT mode)...")
    prune_sparsegpt(args, model, tokenizer, device)

    print("*"*30)
    s_ratio = check_sparsity(model)
    print(f"Sparsity sanity check: {s_ratio*100:.2f}%")
    print("*"*30)

    if args.save_model:
        model.save_pretrained(args.save_model)
        tokenizer.save_pretrained(args.save_model)

if __name__ == '__main__':
    main()

## Change the prune file

In [ ]:
%%writefile lib/prune.py
import torch
import torch.nn as nn
import gc

def find_layers(module, layers=[nn.Linear], name=''):
    if type(module) in layers:
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(find_layers(child, layers=layers, name=name + '.' + name1 if name != '' else name1))
    return res

def prepare_calibration_input(model, dataloader, device):
    layers = model.model.layers
    
    model.model.embed_tokens.to(device)
    if hasattr(model.model, 'rotary_emb'):
        model.model.rotary_emb.to(device)
    
    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros((len(dataloader), model.seqlen, model.config.hidden_size), dtype=dtype, device='cpu')
    
    # We cache metadata needed for subsequent layers
    cache = {'i': 0, 'attention_mask': None, 'position_ids': None, 'position_embeddings': None}

    # FIXED: Added 'output' as the 4th argument
    def hook(module, args, kwargs, output=None):
        inps[cache['i']] = args[0].detach().cpu()
        cache['i'] += 1
        
        if cache['attention_mask'] is None:
            if kwargs.get('attention_mask') is not None:
                cache['attention_mask'] = kwargs.get('attention_mask').detach().cpu()
            if kwargs.get('position_ids') is not None:
                cache['position_ids'] = kwargs.get('position_ids').detach().cpu()
            if 'position_embeddings' in kwargs:
                # Capture Qwen3 specific rotary embeddings
                pe = kwargs['position_embeddings']
                cache['position_embeddings'] = [item.detach().cpu() if isinstance(item, torch.Tensor) else item for item in pe]
        
        raise ValueError("StopForward")

    # Register hook with with_kwargs=True
    handle = layers[0].register_forward_hook(hook, with_kwargs=True)
    layers[0].to(device)

    print("Capturing calibration inputs...")
    for batch in dataloader:
        try:
            with torch.no_grad():
                model(batch[0].to(device))
        except ValueError as e:
            if str(e) == "StopForward":
                continue
            else:
                raise e
            
    handle.remove()
    layers[0].cpu()
    model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()
    
    outs = torch.zeros_like(inps)
    # Returning 5 values for Qwen3 compatibility
    return inps, outs, cache['attention_mask'], cache['position_ids'], cache['position_embeddings']

def prune_sparsegpt(args, model, tokenizer, device):
    from .data import get_loaders
    print("Loading calibration data (C4)")
    dataloader, _ = get_loaders("c4", nsamples=args.nsamples, seed=args.seed, seqlen=model.seqlen, tokenizer=tokenizer)
    
    # Unpack 5 values from the fixed prepare function
    inps, outs, attention_mask, position_ids, pos_embeds = prepare_calibration_input(model, dataloader, device)
    
    if attention_mask is not None: attention_mask = attention_mask.to(device)
    if position_ids is not None: position_ids = position_ids.to(device)
    
    if hasattr(model.model, 'rotary_emb'):
        model.model.rotary_emb.to(device)
        
    layers = model.model.layers

    for i in range(len(layers)):
        print(f"Pruning Layer {i} with SparseGPT...")
        layer = layers[i].to(device)
        subset = find_layers(layer)

        gpt_handlers = {}
        for name in subset:
            gpt_handlers[name] = {
                'H': torch.zeros((subset[name].weight.shape[1], subset[name].weight.shape[1]), device=device, dtype=torch.float32),
                'count': 0
            }

        def get_hook(name):
            def hook(_, inp, out):
                x = inp[0].data
                if len(x.shape) == 3: x = x.reshape((-1, x.shape[-1]))
                x = x.t().float()
                gpt_handlers[name]['H'] += x @ x.t()
                gpt_handlers[name]['count'] += x.shape[1]
            return hook

        handles = []
        for name in subset:
            handles.append(subset[name].register_forward_hook(get_hook(name)))
        
        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                layer_kwargs = {"attention_mask": attention_mask, "position_ids": position_ids}
                
                if pos_embeds is not None:
                    layer_kwargs["position_embeddings"] = [item.to(device) if isinstance(item, torch.Tensor) else item for item in pos_embeds]
                elif hasattr(model.model, 'rotary_emb'):
                    cos, sin = model.model.rotary_emb(curr_in, position_ids)
                    layer_kwargs["position_embeddings"] = (cos, sin)
                
                layer(curr_in, **layer_kwargs)
            
        for h in handles: h.remove()

        for name in subset:
            W = subset[name].weight.data.clone().float()
            H = gpt_handlers[name]['H'] / gpt_handlers[name]['count']
            
            damp = 0.01 * torch.mean(torch.diag(H))
            diag = torch.arange(subset[name].weight.shape[1], device=device)
            H[diag, diag] += damp
            
            Hinv = torch.linalg.inv(H)
            scores = W**2 / (torch.diag(Hinv).reshape((1, -1)))
            
            # Memory-Safe Thresholding for Kaggle
            num_samples = min(1_000_000, scores.numel())
            perm = torch.randperm(scores.numel(), device='cpu')[:num_samples]
            samples = scores.view(-1)[perm.to(device)].to('cpu')
            thresh = torch.quantile(samples, args.sparsity_ratio).item()
            
            mask = scores > thresh
            W_pruned = W.clone()
            W_pruned[~mask] = 0
            subset[name].weight.data = W_pruned.to(subset[name].weight.dtype)
            
            del H, Hinv, scores, W_pruned, samples, perm
            torch.cuda.empty_cache()

        # Update hidden states for the next layer
        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                layer_kwargs = {"attention_mask": attention_mask, "position_ids": position_ids}
                if pos_embeds is not None:
                    layer_kwargs["position_embeddings"] = [item.to(device) if isinstance(item, torch.Tensor) else item for item in pos_embeds]
                elif hasattr(model.model, 'rotary_emb'):
                    cos, sin = model.model.rotary_emb(curr_in, position_ids)
                    layer_kwargs["position_embeddings"] = (cos, sin)
                
                layer_out = layer(curr_in, **layer_kwargs)[0]
                outs[j] = layer_out.detach().cpu()
        
        inps.data = outs.data 
        layers[i] = layer.cpu()
        del layer, gpt_handlers
        gc.collect()
        torch.cuda.empty_cache()

    print("SparseGPT Pruning finished.")

def check_sparsity(model):
    all_weights = 0
    zero_weights = 0
    for name, p in model.named_parameters():
        if 'weight' in name:
            all_weights += p.numel()
            zero_weights += (p == 0).sum().item()
    return float(zero_weights) / all_weights if all_weights > 0 else 0.0

In [ ]:
!touch lib/__init__.py

## Wanda 25

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.25 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_25}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


In [ ]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_25)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_25, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

In [ ]:
# create repo in organization
create_repo(REPO_ID_25, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_25,
    repo_type="model",
    folder_path=OUTPUT_DIR_25,
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

In [ ]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-SGPT-25

## Wanda 50

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.5 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_50}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


In [ ]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_50)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_50, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

In [ ]:
# create repo in organization
create_repo(REPO_ID_50, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_50,
    repo_type="model",
    folder_path=OUTPUT_DIR_50,
    commit_message="Upload SPGT Pruningmodel"
)

print("Upload complete to organization!")

In [ ]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-SGPT-50

## Wanda 70

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.7 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_70}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


In [ ]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_70)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_70, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

In [ ]:
# create repo in organization
create_repo(REPO_ID_70, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_70,
    repo_type="model",
    folder_path=OUTPUT_DIR_70,
    commit_message="Upload SPGT Pruningmodel"
)

print("Upload complete to organization!")

In [ ]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-SGPT-70

## Test the model

In [ ]:
import torch
from transformers import AutoTokenizer

# 1. Setup
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR_70)
# prompt = "what is AI"
prompt = "The best way to prune a neural network is"

# 2. Ensure model is on GPU
model.to("cuda")

# 3. Manually tokenize and move inputs to CUDA
# This ensures the 'index' (input_ids) is on the same device as the weights
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate using the model directly (more reliable than pipeline here)
with torch.no_grad():
    output_ids = model.generate(
        **inputs, 
        max_new_tokens=50, 
        do_sample=True, 
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Decode and Print
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)

In [ ]:
# Step 1: The Perplexity Script

import torch
from tqdm import tqdm
from datasets import load_dataset

def evaluate_perplexity(model, tokenizer, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1"):
    # 1. Load the test set
    test_data = load_dataset(dataset_name, dataset_config, split="test")
    encodings = tokenizer("\n\n".join(test_data["text"]), return_tensors="pt")

    max_length = 2048 # Llama 3.2 context window
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    
    print(f"Calculating Perplexity for {seq_len} tokens...")
    
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            # loss is calculated using CrossEntropyLoss which averages over valid labels
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

# Run the evaluation
ppl_value = evaluate_perplexity(model, tokenizer)
print(f"\nFinal Perplexity: {ppl_value:.4f}")

In [ ]:
import time

def mobile_sim_benchmark(model, tokenizer, prompt="Explain AI in 10 words."):
    # 1. Force CPU and limit threads to simulate mobile processor
    torch.set_num_threads(4) 
    model.to("cpu")
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Warm start
    _ = model.generate(**inputs, max_new_tokens=1)
    
    # Start Timing
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=30, do_sample=True)
    end = time.time()
    
    # Calculate Metrics
    tokens_gen = len(outputs[0]) - len(inputs[0])
    tps = tokens_gen / (end - start)
    
    print(f"--- Mobile Simulation Result ---")
    print(f"Tokens/Sec (TPS): {tps:.2f}")
    print(f"Latency: {end - start:.2f}s")
    
    if tps < 5:
        print("Status: Too slow for phone (Needs 4-bit Quantization)")
    else:
        print("Status: Good for Edge!")

# mobile_sim_benchmark(model, tokenizer)

In [ ]:
# !python main.py \
#     --model "meta-llama/Llama-3.2-3B-Instruct" \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --save out/llama_3.2_pruned/

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.5 \
#     --sparsity_type unstructured \
#     --save_model out/llama_3.2_pruned/

# Important Kaggle Tip: Since you have cuda:0 detected now, the process will be much faster. 
# However, Llama 3.2 3B uses a lot of memory during the "Wanda" phase (where it calculates the $X^TX$ Hessian matrix). 
# If you hit an Out of Memory (OOM) error, reduce the --nsamples from the default 128 to 64.

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --nsamples 32 \
#     --save out/llama_3_2_pruned/